# LLaDA — INT8 vs INT4 Test
### Loads directly from Google Drive — No re-quantization needed
- INT8 path: `/content/drive/MyDrive/LLaDA-Quantized/weights/llada_int8_quantized.pt`
- INT4 path: `/content/drive/MyDrive/LLaDA-Quantized/weights/llada_int4_quantized.pt`

In [ ]:
# Cell 1 — Install
!pip install transformers==4.40.0 accelerate huggingface_hub psutil -q
print('Install done!')

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

INT8_PATH = '/content/drive/MyDrive/LLaDA-Quantized/weights/llada_int8_quantized.pt'
INT4_PATH = '/content/drive/MyDrive/LLaDA-Quantized/weights/llada_int4_quantized.pt'

# Verify files exist
print('Checking files...')
print(f'INT8: {INT8_PATH}')
print(f'  Exists: {os.path.exists(INT8_PATH)}')
if os.path.exists(INT8_PATH):
    print(f'  Size: {os.path.getsize(INT8_PATH)/1e9:.2f} GB')

print(f'INT4: {INT4_PATH}')
print(f'  Exists: {os.path.exists(INT4_PATH)}')
if os.path.exists(INT4_PATH):
    print(f'  Size: {os.path.getsize(INT4_PATH)/1e9:.2f} GB')

In [ ]:
# Cell 3 — Imports + Device + Memory Utils
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import time
import psutil

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    elif torch.backends.mps.is_available():
        return 'mps'
    else:
        return 'cpu'

device = get_device()

def get_memory_stats():
    ram = psutil.virtual_memory()
    stats = {
        'ram_used_gb': ram.used / 1e9,
        'ram_total_gb': ram.total / 1e9,
        'ram_percent': ram.percent,
        'ram_available_gb': ram.available / 1e9,
    }
    if device == 'cuda':
        stats['gpu_allocated_gb'] = torch.cuda.memory_allocated() / 1e9
        stats['gpu_reserved_gb'] = torch.cuda.memory_reserved() / 1e9
        stats['gpu_total_gb'] = torch.cuda.get_device_properties(0).total_memory / 1e9
    return stats

def print_memory(label=''):
    s = get_memory_stats()
    print(f'[{label}]')
    print(f'  RAM: {s["ram_used_gb"]:.2f} GB used / {s["ram_total_gb"]:.1f} GB total ({s["ram_percent"]}%)')
    print(f'  RAM available: {s["ram_available_gb"]:.2f} GB')
    if 'gpu_allocated_gb' in s:
        print(f'  GPU allocated: {s["gpu_allocated_gb"]:.2f} GB')
        print(f'  GPU reserved:  {s["gpu_reserved_gb"]:.2f} GB')
        print(f'  GPU total:     {s["gpu_total_gb"]:.2f} GB')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
print_memory('Initial')

In [ ]:
# Cell 4 — Load Tokenizer (shared)
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    'GSAI-ML/LLaDA-8B-Instruct',
    trust_remote_code=True
)
print('Tokenizer ready!')

In [ ]:
# Cell 5 — Generation Function (shared by both)
@torch.no_grad()
def generate_with_stats(model, tokenizer, prompt, steps=64, gen_length=64):
    MASK_ID = 126336

    t0 = time.time()
    messages = [{'role': 'user', 'content': prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True
    ).to(device)
    prompt_len = input_ids.shape[1]
    x = torch.full(
        (1, prompt_len + gen_length),
        MASK_ID, dtype=torch.long, device=device
    )
    x[:, :prompt_len] = input_ids
    setup_time = time.time() - t0

    t1 = time.time()
    for step in range(steps):
        logits = model(x).logits
        masked = (x == MASK_ID)
        if masked.sum() == 0:
            break
        probs = torch.softmax(logits, dim=-1)
        confidence, predicted = probs.max(dim=-1)
        confidence[~masked] = -float('inf')
        num_unmask = max(1, masked.sum().item() // max(1, steps - step))
        top_pos = confidence[0].topk(num_unmask).indices
        x[0, top_pos] = predicted[0, top_pos]
    decode_time = time.time() - t1

    out_str = tokenizer.decode(x[0, prompt_len:], skip_special_tokens=True)
    return out_str, {
        'prompt': prompt,
        'prompt_tokens': prompt_len,
        'generated_tokens': gen_length,
        'steps': steps,
        'setup_time': round(setup_time, 4),
        'decode_time': round(decode_time, 4),
        'total_time': round(setup_time + decode_time, 4),
        'tokens_per_sec': round(gen_length / decode_time, 2) if decode_time > 0 else 0,
        'steps_per_sec': round(steps / decode_time, 2) if decode_time > 0 else 0,
        'ms_per_token': round((decode_time / gen_length) * 1000, 2) if gen_length > 0 else 0,
    }

print('Generation function ready!')

---
## Part 1 — INT8 Test

In [ ]:
# Cell 6 — INT8 Class Definition
class Int8Linear(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        w = original_layer.weight.data.float()
        scale = w.abs().max(dim=1, keepdim=True).values / 127.0
        scale = scale.clamp(min=1e-8)
        w_int8 = (w / scale).round().clamp(-128, 127).to(torch.int8)
        self.register_buffer('weight_int8', w_int8)
        self.register_buffer('scale', scale.to(torch.float32))
        if original_layer.bias is not None:
            self.register_buffer('bias', original_layer.bias.data.to(torch.float32))
        else:
            self.bias = None
        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

    def forward(self, x):
        w = self.weight_int8.to(torch.float32)
        w = w * self.scale
        w = w.to(x.dtype)
        bias = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w, bias)


def build_int8_structure(model):
    """Replace Linear layers with Int8Linear — structure only, no computation"""
    replaced = 0
    def replace_recursive(module):
        nonlocal replaced
        for child_name, child in module.named_children():
            if isinstance(child, nn.Linear):
                setattr(module, child_name, Int8Linear(child))
                replaced += 1
            else:
                replace_recursive(child)
    replace_recursive(model)
    return model, replaced

print('Int8Linear class ready!')

In [ ]:
# Cell 7 — Load INT8 from Google Drive
print('='*55)
print('INT8 — Loading from Google Drive')
print('='*55)
print_memory('Before INT8 load')

t_start = time.time()

# Step 1: Architecture
print('\nStep 1: Loading architecture on CPU...')
model_int8 = AutoModel.from_pretrained(
    'GSAI-ML/LLaDA-8B-Instruct',
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    trust_remote_code=True
)
print_memory('After architecture load')

# Step 2: Build INT8 structure
print('\nStep 2: Building INT8 structure...')
model_int8, int8_layer_count = build_int8_structure(model_int8)
print(f'  INT8 layers created: {int8_layer_count}')

# Step 3: Load weights from Drive
print(f'\nStep 3: Loading weights from Drive...')
print(f'  Path: {INT8_PATH}')
state_dict_int8 = torch.load(INT8_PATH, map_location='cpu')
model_int8.load_state_dict(state_dict_int8)
del state_dict_int8
print('  Weights loaded!')

# Step 4: Move to device
print(f'\nStep 4: Moving to {device}...')
model_int8 = model_int8.to(device)
model_int8.eval()

t_load = time.time() - t_start

# Verify
int8_verified = 0
for name, module in model_int8.named_modules():
    if isinstance(module, Int8Linear):
        int8_verified += 1

print('\n' + '='*55)
print('INT8 LOAD COMPLETE')
print('='*55)
print(f'Load time:         {t_load:.2f}s')
print(f'INT8 layers:       {int8_verified}')
print(f'Weight dtype:      torch.int8')
print_memory('Final INT8 state')

In [ ]:
# Cell 8 — INT8 Generation Test
prompts = [
    'Explain quantum computing in 2 short sentences.',
    'Bharat ki rajdhani kya hai?',
    'What is machine learning?',
    'AI kya hota hai?',
]

int8_results = []

print('='*55)
print('INT8 GENERATION TEST')
print(f'Device: {device}')
print('='*55)

for prompt in prompts:
    out, stats = generate_with_stats(model_int8, tokenizer, prompt)
    int8_results.append({'output': out, 'stats': stats})

    print(f'\nQ: {prompt}')
    print(f'A: {out}')
    print(f'Prompt tokens:    {stats["prompt_tokens"]}')
    print(f'Generated tokens: {stats["generated_tokens"]}')
    print(f'Setup time:       {stats["setup_time"]}s')
    print(f'Decode time:      {stats["decode_time"]}s')
    print(f'Total time:       {stats["total_time"]}s')
    print(f'Speed:            {stats["tokens_per_sec"]} tok/s')
    print(f'Steps/sec:        {stats["steps_per_sec"]}')
    print(f'Ms per token:     {stats["ms_per_token"]}ms')
    print('-'*40)

print_memory('After INT8 generation')

In [ ]:
# Cell 9 — Free INT8 memory
del model_int8
import gc
gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()
print('INT8 model freed!')
print_memory('After INT8 free')

---
## Part 2 — INT4 Test

In [ ]:
# Cell 10 — INT4 Class Definition
class Int4LinearPacked(nn.Module):
    def __init__(self, original_layer):
        super().__init__()
        w = original_layer.weight.data.float()
        scale = w.abs().max(dim=1, keepdim=True).values / 7.0
        scale = scale.clamp(min=1e-8)
        w_int4 = (w / scale).round().clamp(-8, 7).to(torch.int8)
        w_flat = w_int4.reshape(-1)
        if w_flat.shape[0] % 2 != 0:
            w_flat = torch.cat([w_flat, torch.zeros(1, dtype=torch.int8)])
        even = w_flat[0::2] & 0x0F
        odd  = (w_flat[1::2] & 0x0F) << 4
        packed = (even | odd).to(torch.uint8)
        self.register_buffer('weight_packed', packed)
        self.register_buffer('scale', scale.to(torch.float32))
        self.original_shape = w_int4.shape
        if original_layer.bias is not None:
            self.register_buffer('bias', original_layer.bias.data.to(torch.float32))
        else:
            self.bias = None
        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

    def forward(self, x):
        packed = self.weight_packed
        result = torch.empty(packed.numel() * 2, dtype=torch.int8, device=packed.device)
        result[0::2] = (packed & 0x0F).to(torch.int8)
        result[1::2] = ((packed >> 4) & 0x0F).to(torch.int8)
        mask = result > 7
        result[mask] = result[mask] - 16
        numel = self.original_shape[0] * self.original_shape[1]
        w = result[:numel].reshape(self.original_shape).float()
        w = w * self.scale
        w = w.to(x.dtype)
        bias = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w, bias)


def build_int4_structure(model):
    """Replace Linear layers with Int4LinearPacked — structure only"""
    replaced = 0
    def replace_recursive(module):
        nonlocal replaced
        for child_name, child in module.named_children():
            if isinstance(child, nn.Linear):
                setattr(module, child_name, Int4LinearPacked(child))
                replaced += 1
            else:
                replace_recursive(child)
    replace_recursive(model)
    return model, replaced

print('Int4LinearPacked class ready!')

In [ ]:
# Cell 11 — Load INT4 from Google Drive
print('='*55)
print('INT4 — Loading from Google Drive')
print('='*55)
print_memory('Before INT4 load')

t_start = time.time()

# Step 1: Architecture
print('\nStep 1: Loading architecture on CPU...')
model_int4 = AutoModel.from_pretrained(
    'GSAI-ML/LLaDA-8B-Instruct',
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    trust_remote_code=True
)
print_memory('After architecture load')

# Step 2: Build INT4 structure
print('\nStep 2: Building INT4 structure...')
model_int4, int4_layer_count = build_int4_structure(model_int4)
print(f'  INT4 layers created: {int4_layer_count}')

# Step 3: Load weights from Drive
print(f'\nStep 3: Loading weights from Drive...')
print(f'  Path: {INT4_PATH}')
state_dict_int4 = torch.load(INT4_PATH, map_location='cpu')
model_int4.load_state_dict(state_dict_int4)
del state_dict_int4
print('  Weights loaded!')

# Step 4: Move to device
print(f'\nStep 4: Moving to {device}...')
model_int4 = model_int4.to(device)
model_int4.eval()

t_load = time.time() - t_start

# Verify
int4_verified = 0
for name, module in model_int4.named_modules():
    if isinstance(module, Int4LinearPacked):
        int4_verified += 1

print('\n' + '='*55)
print('INT4 LOAD COMPLETE')
print('='*55)
print(f'Load time:         {t_load:.2f}s')
print(f'INT4 layers:       {int4_verified}')
print(f'Weight dtype:      torch.uint8 (packed)')
print_memory('Final INT4 state')

In [ ]:
# Cell 12 — INT4 Generation Test
int4_results = []

print('='*55)
print('INT4 GENERATION TEST')
print(f'Device: {device}')
print('='*55)

for prompt in prompts:
    out, stats = generate_with_stats(model_int4, tokenizer, prompt)
    int4_results.append({'output': out, 'stats': stats})

    print(f'\nQ: {prompt}')
    print(f'A: {out}')
    print(f'Prompt tokens:    {stats["prompt_tokens"]}')
    print(f'Generated tokens: {stats["generated_tokens"]}')
    print(f'Setup time:       {stats["setup_time"]}s')
    print(f'Decode time:      {stats["decode_time"]}s')
    print(f'Total time:       {stats["total_time"]}s')
    print(f'Speed:            {stats["tokens_per_sec"]} tok/s')
    print(f'Steps/sec:        {stats["steps_per_sec"]}')
    print(f'Ms per token:     {stats["ms_per_token"]}ms')
    print('-'*40)

print_memory('After INT4 generation')

---
## Part 3 — Final Comparison

In [ ]:
# Cell 13 — Full Comparison Table
print('='*60)
print('FINAL COMPARISON — INT8 vs INT4')
print(f'Device: {device}')
print('='*60)

# Per prompt comparison
for i, prompt in enumerate(prompts):
    r8 = int8_results[i]
    r4 = int4_results[i]
    print(f'\nPrompt: {prompt}')
    print(f'  INT8 output: {r8["output"][:80]}...' if len(r8['output']) > 80 else f'  INT8: {r8["output"]}')
    print(f'  INT4 output: {r4["output"][:80]}...' if len(r4['output']) > 80 else f'  INT4: {r4["output"]}')
    print(f'  INT8 speed: {r8["stats"]["tokens_per_sec"]} tok/s | INT4 speed: {r4["stats"]["tokens_per_sec"]} tok/s')

# Summary stats
avg_int8_speed   = sum(r['stats']['tokens_per_sec'] for r in int8_results) / len(int8_results)
avg_int4_speed   = sum(r['stats']['tokens_per_sec'] for r in int4_results) / len(int4_results)
avg_int8_time    = sum(r['stats']['total_time'] for r in int8_results) / len(int8_results)
avg_int4_time    = sum(r['stats']['total_time'] for r in int4_results) / len(int4_results)
avg_int8_ms_tok  = sum(r['stats']['ms_per_token'] for r in int8_results) / len(int8_results)
avg_int4_ms_tok  = sum(r['stats']['ms_per_token'] for r in int4_results) / len(int4_results)

print('\n' + '='*60)
print(f'{"Metric":<28} {"INT8":>14} {"INT4":>14}')
print('-'*58)
print(f'{"Avg speed (tok/s)":<28} {avg_int8_speed:>14.2f} {avg_int4_speed:>14.2f}')
print(f'{"Avg total time (s)":<28} {avg_int8_time:>14.2f} {avg_int4_time:>14.2f}')
print(f'{"Avg ms per token":<28} {avg_int8_ms_tok:>14.2f} {avg_int4_ms_tok:>14.2f}')
print(f'{"Layers quantized":<28} {"225":>14} {"225":>14}')
print(f'{"Weight dtype":<28} {"torch.int8":>14} {"torch.uint8":>14}')
print(f'{"Drive file size":<28} {"8.54 GB":>14} {"4.46 GB":>14}')
print(f'{"Device":<28} {device:>14} {device:>14}')
print('='*60)

if avg_int4_speed > 0:
    speedup = avg_int8_speed / avg_int4_speed
    print(f'INT8 is {speedup:.2f}x faster than INT4 on {device}')

print_memory('Final state')